<a href="https://colab.research.google.com/github/springboardmentor787-stack/Company-Internal-Chatbot-with-Role-Based-Access-Control-RBAC---Group-1/blob/Srija-Mitra/MileStone2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
pip install fastapi streamlit langchain-community sentence-transformers chromadb pandas langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4

In [3]:
!git clone https://github.com/springboardmentor441p-coderr/Fintech-data.git Fintech-data-main

Cloning into 'Fintech-data-main'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 43 (delta 2), reused 17 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 51.89 KiB | 3.99 MiB/s, done.
Resolving deltas: 100% (2/2), done.


In [4]:
!ls Fintech-data-main


engineering  Finance  general  HR  marketing


In [5]:
role_access_mapping = {
    "Finance": ["Finance", "C-Level"],
    "marketing": ["Marketing", "C-Level"],
    "HR": ["HR", "C-Level"],
    "engineering": ["Engineering", "C-Level"],
    "general": ["Finance", "Marketing", "HR", "Engineering", "C-Level", "General"]
}


In [6]:
from langchain_community.document_loaders import TextLoader, CSVLoader
from pathlib import Path

file_path = Path("Fintech-data-main") / "engineering" / "engineering_master_doc.md"
loader = TextLoader(str(file_path), encoding="utf-8", autodetect_encoding=True)


docs = loader.load()
print(docs)

[Document(metadata={'source': 'Fintech-data-main/engineering/engineering_master_doc.md'}, page_content='# FinSolve Technologies Engineering Document\n\n## 1. Introduction\n\n### 1.1 Company Overview\nFinSolve Technologies is a leading FinTech company headquartered in Bangalore, India, with operations across North America, Europe, and Asia-Pacific. Founded in 2018, FinSolve provides innovative financial solutions, including digital banking, payment processing, wealth management, and enterprise financial analytics, serving over 2 million individual users and 10,000 businesses globally.\n\n### 1.2 Purpose\nThis engineering document outlines the technical architecture, development processes, and operational guidelines for FinSolve\'s product ecosystem. It serves as a comprehensive guide for engineering teams, stakeholders, and partners to ensure alignment with FinSolve\'s mission: "To empower financial freedom through secure, scalable, and innovative technology solutions."\n\n### 1.3 Scope

In [9]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader, CSVLoader

DATA_ROOT = Path("Fintech-data-main")
all_documents = []

for department, roles in role_access_mapping.items():
    dept_path = DATA_ROOT / department

    for file in dept_path.glob("*"):


        if file.suffix == ".md":
            loader = TextLoader(str(file), encoding="utf-8")
        elif file.suffix == ".csv":
            loader = CSVLoader(str(file))
        else:
            continue

        docs = loader.load()

        for d in docs:
            d.page_content = clean_text(d.page_content)


        for d in docs:
            d.metadata = {
                "department": department,
                "allowed_roles": roles,
                "source": file.name
            }

        all_documents.extend(docs)

print("Total loaded documents:", len(all_documents))
print("Sample metadata:", all_documents[0].metadata)

Total loaded documents: 109
Sample metadata: {'department': 'Finance', 'allowed_roles': ['Finance', 'C-Level'], 'source': 'financial_summary.md'}


In [8]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9\s.,]", "", text)
    return text.strip()


In [10]:
import langchain
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ".", " "],
    chunk_size=500,
    chunk_overlap=50,
    length_function=len
)

chunks = text_splitter.split_documents(all_documents)

print("Total chunks:", len(chunks))
print("Sample chunk metadata:", chunks[0].metadata)

Total chunks: 315
Sample chunk metadata: {'department': 'Finance', 'allowed_roles': ['Finance', 'C-Level'], 'source': 'financial_summary.md'}


In [11]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


for chunk in chunks:
    if isinstance(chunk.metadata.get('allowed_roles'), list):
        chunk.metadata['allowed_roles'] = ",".join(chunk.metadata['allowed_roles'])


vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)

/tmp/ipython-input-182880686.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
!ls chroma_db

cc2ce87a-dd9d-41eb-9dfd-a387f1b479c2  chroma.sqlite3


In [13]:

query = "What is our quarterly revenue and financial performance?"

hr_filter = {
    "allowed_roles": { "$eq": "HR" }
}

results = vector_db.similarity_search(query, k=3, filter=hr_filter)

print(f"Query: {query}")
print(f"User Role: HR")
print(f"Results found: {len(results)}")

if len(results) == 0:
    print("Access Denied. HR users cannot see Finance data.")
else:
    print("Sensitive data was leaked!")

Query: What is our quarterly revenue and financial performance?
User Role: HR
Results found: 0
Access Denied. HR users cannot see Finance data.


In [14]:

finance_filter = {
    "allowed_roles": { "$eq": "Finance,C-Level" }
}

authorized_results = vector_db.similarity_search(query, k=3, filter=finance_filter)

print(f"\nQuery: {query}")
print(f"User Role: Finance")
print(f"Results found: {len(authorized_results)}")

if len(authorized_results) > 0:
    print("Data retrieved for authorized user.")


Query: What is our quarterly revenue and financial performance?
User Role: Finance
Results found: 3
Data retrieved for authorized user.


In [15]:
test_roles = ["HR", "Engineering", "Marketing", "Finance", "C-Level", "General"]
finance_query = "What is our quarterly revenue and financial performance?"

print(f"--- SECURITY AUDIT: {finance_query} ---")

role_to_allowed_strings = {}
all_unique_allowed_role_metadata_strings = set()
for roles_list_from_mapping in role_access_mapping.values():
    all_unique_allowed_role_metadata_strings.add(",".join(roles_list_from_mapping))

for single_role_to_test in test_roles:
    matching_allowed_strings = []
    for allowed_role_metadata_str in all_unique_allowed_role_metadata_strings:
        if single_role_to_test in allowed_role_metadata_str.split(','):
            if single_role_to_test not in ["Finance", "C-Level"] and "Finance" in allowed_role_metadata_str.split(','):
                continue
            matching_allowed_strings.append(allowed_role_metadata_str)
    role_to_allowed_strings[single_role_to_test] = matching_allowed_strings

for role in test_roles:
    allowed_strings_for_this_role = role_to_allowed_strings.get(role, [])

    if role not in ["Finance", "C-Level"]:
        allowed_strings_for_this_role = ["__NO_MATCH_POSSIBLE__"]

    if not allowed_strings_for_this_role:
        role_filter = {"allowed_roles": {"$in": ["__NO_MATCH_PLACEHOLDER__"]}}
    else:
        role_filter = {"allowed_roles": {"$in": allowed_strings_for_this_role}}

    test_results = vector_db.similarity_search(finance_query, k=3, filter=role_filter)

    expected_to_find_results = (role == "Finance" or role == "C-Level")

    if expected_to_find_results:
        status = "ACCESS GRANTED" if len(test_results) > 0 else "FAILURE: No data found for authorized role"
    else:
        status = "ACCESS DENIED" if len(test_results) == 0 else "SECURITY LEAK DETECTED"

    print(f"Role: {role:<12} | Results: {len(test_results)} | Status: {status}")

--- SECURITY AUDIT: What is our quarterly revenue and financial performance? ---
Role: HR           | Results: 0 | Status: ACCESS DENIED
Role: Engineering  | Results: 0 | Status: ACCESS DENIED
Role: Marketing    | Results: 0 | Status: ACCESS DENIED
Role: Finance      | Results: 3 | Status: ACCESS GRANTED
Role: C-Level      | Results: 3 | Status: ACCESS GRANTED
Role: General      | Results: 0 | Status: ACCESS DENIED


# MILESTONE 2

In [16]:
import re

def preprocess_query(query):
    query = query.lower()
    query = re.sub(r"[^a-z0-9\s]", "", query)
    query = re.sub(r"\s+", " ", query).strip()
    return query

In [17]:
def python_rbac_filter(docs, user_role):
    allowed_docs = []
    blocked_docs = []

    for doc in docs:
        allowed_roles_str = doc.metadata.get("allowed_roles", "")
        allowed_roles = [r.strip() for r in allowed_roles_str.split(",")]

        if user_role in allowed_roles:
            allowed_docs.append(doc)
        else:
            blocked_docs.append(doc)

    return allowed_docs, blocked_docs

In [18]:
def log_chunks(query, role, raw_docs, allowed_docs, blocked_docs):
    print("\n" + "=" * 70)
    print("QUERY:", query)
    print("ROLE:", role)

    print("\n--- RAW RETRIEVED CHUNKS ---")
    for i, doc in enumerate(raw_docs):
        print(f"\n[RAW {i+1}]")
        print("Source:", doc.metadata.get("source"))
        print("Department:", doc.metadata.get("department"))
        print("Allowed Roles:", doc.metadata.get("allowed_roles"))
        print("Content:", doc.page_content[:300])

    print("\n--- BLOCKED BY RBAC ---")
    for i, doc in enumerate(blocked_docs):
        print(f"\n[BLOCKED {i+1}]")
        print("Source:", doc.metadata.get("source"))
        print("Department:", doc.metadata.get("department"))
        print("Allowed Roles:", doc.metadata.get("allowed_roles"))

    print("\n--- FINAL ALLOWED CHUNKS ---")
    for i, doc in enumerate(allowed_docs):
        print(f"\n[ALLOWED {i+1}]")
        print("Source:", doc.metadata.get("source"))
        print("Department:", doc.metadata.get("department"))
        print("Allowed Roles:", doc.metadata.get("allowed_roles"))
        print("Content:", doc.page_content[:300])

    print("=" * 70 + "\n")



In [22]:
def run_role_based_query():
    query = input("Enter your query: ").strip()
    role = input("Enter your role (HR / Engineering / Finance / Marketing / C-Level/ General): ").strip()

    if not query or not role:
        print("Query and role are required.")
        return

    query = preprocess_query(query)

    raw_docs = vector_db.similarity_search(query, k=5)

    allowed_docs, blocked_docs = python_rbac_filter(raw_docs, role)

    log_chunks(query, role, raw_docs, allowed_docs, blocked_docs)

    if not allowed_docs:
        print("ACCESS DENIED OR NO RELEVANT DATA FOUND")
        return

    print("\n--- FINAL ANSWER CONTEXT ---")
    for doc in allowed_docs:
        print(doc.page_content[:300])

In [23]:
run_role_based_query()

Enter your query: What is our quarterly revenue?
Enter your role (HR / Engineering / Finance / Marketing / C-Level/ General): Finance

QUERY: what is our quarterly revenue
ROLE: Finance

--- RAW RETRIEVED CHUNKS ---

[RAW 1]
Source: quarterly_financial_report.md
Department: Finance
Allowed Roles: Finance,C-Level
Content: . key financial highlights include  revenue 2.1 billion, up 22 yoy, driven by strong customer acquisition and increased transaction volumes.  gross margin 58, reflecting effective cost management and pricing strategies.  operating income 500 million, supported by streamlined operations and highmargi

[RAW 2]
Source: quarterly_financial_report.md
Department: Finance
Allowed Roles: Finance,C-Level
Content: .  net income 325 million, up 18 yoy, driven by topline growth and margin expansion.  marketing spend 650 million, allocated to endofyear promotions and b2b marketing campaigns.  vendor costs 135 million, increased due to high sales volume during holiday campaigns.  qu

In [24]:
run_role_based_query()

Enter your query: What is our quarterly revenue?
Enter your role (HR / Engineering / Finance / Marketing / C-Level/ General): HR

QUERY: what is our quarterly revenue
ROLE: HR

--- RAW RETRIEVED CHUNKS ---

[RAW 1]
Source: quarterly_financial_report.md
Department: Finance
Allowed Roles: Finance,C-Level
Content: . key financial highlights include  revenue 2.1 billion, up 22 yoy, driven by strong customer acquisition and increased transaction volumes.  gross margin 58, reflecting effective cost management and pricing strategies.  operating income 500 million, supported by streamlined operations and highmargi

[RAW 2]
Source: quarterly_financial_report.md
Department: Finance
Allowed Roles: Finance,C-Level
Content: .  net income 325 million, up 18 yoy, driven by topline growth and margin expansion.  marketing spend 650 million, allocated to endofyear promotions and b2b marketing campaigns.  vendor costs 135 million, increased due to high sales volume during holiday campaigns.  quarterly ex

In [25]:
run_role_based_query()

Enter your query: What is our quarterly revenue?
Enter your role (HR / Engineering / Finance / Marketing / C-Level/ General): C-Level

QUERY: what is our quarterly revenue
ROLE: C-Level

--- RAW RETRIEVED CHUNKS ---

[RAW 1]
Source: quarterly_financial_report.md
Department: Finance
Allowed Roles: Finance,C-Level
Content: . key financial highlights include  revenue 2.1 billion, up 22 yoy, driven by strong customer acquisition and increased transaction volumes.  gross margin 58, reflecting effective cost management and pricing strategies.  operating income 500 million, supported by streamlined operations and highmargi

[RAW 2]
Source: quarterly_financial_report.md
Department: Finance
Allowed Roles: Finance,C-Level
Content: .  net income 325 million, up 18 yoy, driven by topline growth and margin expansion.  marketing spend 650 million, allocated to endofyear promotions and b2b marketing campaigns.  vendor costs 135 million, increased due to high sales volume during holiday campaigns.  qu